# Notebook 02 — Data Quality Checking & Cleaning
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 3 of 8  
**Objective:** Systematically check the dataset for quality issues (missing values, duplicates, invalid values, outliers), apply a cleaning pipeline, and produce a clean DataFrame ready for EDA.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_data
from src.data_cleaner import check_data_quality, print_quality_report, clean_data
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN, NUMERIC_FEATURES,
    CATEGORICAL_FEATURES, BINARY_FEATURES, FIGURES_DIR, TABLES_DIR
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

---
## 1. Load Raw Data

In [ ]:
df_raw = load_data(RAW_DATA_PATH)
print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

---
## 2. Data Quality Report

In [ ]:
print_quality_report(df_raw)

---
## 3. Missing Value Heatmap

In [ ]:
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e05c5c' if v > 0 else '#b6d7a8' for v in missing_pct.values]
ax.bar(missing_pct.index, missing_pct.values, color=colors, edgecolor='white')
ax.set_title('Missing Value Percentage by Column', fontweight='bold')
ax.set_ylabel('Missing (%)')
ax.set_ylim(0, max(missing_pct.max() + 1, 5))
ax.tick_params(axis='x', rotation=45)
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/missing_values.png')

---
## 4. Outlier Detection — Box Plots (Before Cleaning)

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(NUMERIC_FEATURES) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    axes[i].boxplot(
        df_raw[col].dropna(),
        vert=True,
        patch_artist=True,
        boxprops=dict(facecolor='#4a90d9', alpha=0.6),
        medianprops=dict(color='#c0392b', linewidth=2),
        flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#e05c5c')
    )
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel(col)

for j in range(len(NUMERIC_FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Detection — Box Plots (Raw Data)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'outlier_boxplots_raw.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/outlier_boxplots_raw.png')

---
## 5. IQR-Based Outlier Summary

In [ ]:
outlier_summary = []
for col in NUMERIC_FEATURES:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    lo = Q1 - 1.5 * IQR
    hi = Q3 + 1.5 * IQR
    n_outliers = int(((df_raw[col] < lo) | (df_raw[col] > hi)).sum())
    outlier_summary.append({
        'Feature':      col,
        'Q1':           round(Q1, 2),
        'Q3':           round(Q3, 2),
        'IQR':          round(IQR, 2),
        'Lower Fence':  round(lo, 2),
        'Upper Fence':  round(hi, 2),
        'Outliers':     n_outliers,
        'Outlier %':    round(n_outliers / len(df_raw) * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_summary).set_index('Feature')
outlier_df.to_csv(TABLES_DIR / 'outlier_summary.csv')
print('Saved -> outputs/tables/outlier_summary.csv')
outlier_df

---
## 6. Categorical Value Validation

In [ ]:
from src.data_cleaner import VALID_VALUES

print('Categorical value validation:')
all_ok = True
for col, valid_set in VALID_VALUES.items():
    actual = set(df_raw[col].astype(str).str.strip().unique())
    unexpected = actual - valid_set
    status = 'OK' if not unexpected else f'ISSUE: {unexpected}'
    print(f'  {col:<20} {status}')
    if unexpected:
        all_ok = False

print()
print('All categorical values valid:', all_ok)

---
## 7. Apply the Cleaning Pipeline

In [ ]:
df_clean = clean_data(df_raw)
print(f'\nClean dataset shape: {df_clean.shape}')
df_clean.head()

---
## 8. Before vs After — Row Count Comparison

In [ ]:
comparison = pd.DataFrame({
    'Stage':  ['Raw', 'After Cleaning'],
    'Rows':   [len(df_raw), len(df_clean)],
})

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(comparison['Stage'], comparison['Rows'],
              color=['#4a90d9', '#27ae60'], edgecolor='white', width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{int(bar.get_height()):,}", ha='center', fontsize=10)
ax.set_title('Row Count: Raw vs Clean Dataset', fontweight='bold')
ax.set_ylabel('Number of Rows')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'before_after_cleaning.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/before_after_cleaning.png')

---
## 9. Inspect Cleaned Dtypes

In [ ]:
dtype_df = pd.DataFrame({
    'Column':   df_clean.columns,
    'Dtype':    df_clean.dtypes.values.astype(str),
    'NonNull':  df_clean.notnull().sum().values,
    'Unique':   df_clean.nunique().values,
    'Sample':   [df_clean[c].iloc[0] for c in df_clean.columns]
})
dtype_df

---
## 10. Post-Clean Quality Check

In [ ]:
# Re-add a dummy LoanID column just for the quality checker
df_check = df_clean.copy()
df_check['LoanID'] = range(len(df_check))

from src.data_cleaner import check_data_quality
report = check_data_quality(df_check)

print('Post-clean quality check:')
print(f'  Issues found : {report["total_issues_found"]}')
print(f'  QA passed    : {report["data_quality_passed"]}')

---
## 11. Correlation Matrix — Numeric Features (Clean Data)

In [ ]:
num_clean = df_clean[NUMERIC_FEATURES + [TARGET_COLUMN]]
corr = num_clean.corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix — Numeric Features + Target', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/correlation_matrix.png')

---
## 12. Save Clean Dataset

In [ ]:
from src.config import DATA_DIR
out_path = DATA_DIR / 'Loan_default_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Clean dataset saved -> {out_path}')
print(f'Shape: {df_clean.shape}')

---
## 13. Quality Summary

| Check | Result |
|---|---|
| Missing values | *(fill after run)* |
| Duplicate rows removed | *(fill after run)* |
| Invalid categoricals | *(fill after run)* |
| Out-of-range numerics | *(fill after run)* |
| Binary encoding applied | Yes — Yes/No -> 1/0 |
| Numeric clipping applied | Yes — domain bounds |
| LoanID column dropped | Yes |
| Final clean shape | *(fill after run)* |

---
**Next:** Notebook 03 — Exploratory Data Analysis